# 01 Scalar Case Diagnostics

I start with the shortlist because it is the quickest way to see whether the scalar twin is producing physically plausible beams for the writing targets. The summary now shows both the legacy peak-trace Bessel zone and the stricter stable Bessel region, so I can immediately see when the old z metric was being generous.

In [1]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vbb_study import setup_study, vbb_style

PATHS = setup_study.bootstrap(Path.cwd())
ROOT = PATHS["root"]
PUB_ROOT = PATHS["publication"]
LAB_ROOT = PATHS["modular_lab"]

import bessel_twin_core as bt
from Publication_Study import publication_diagnostics as pdiag

In [2]:
PRESET = "fast"
PATH = "realistic"
OUTPUTS = bt.ensure_output_tree(PATHS["outputs"])
PUB_FIG_DIR = OUTPUTS["figures"] / "publication_study"
PUB_CSV_DIR = OUTPUTS["csv"] / "publication_study"
PUB_FIG_DIR.mkdir(parents=True, exist_ok=True)
PUB_CSV_DIR.mkdir(parents=True, exist_ok=True)
base = bt.default_config(PRESET)

## Shortlist Comparison

Reuse the experiment-planning shortlist, but run the realistic device-to-sample path so the publication figures reflect the lab-facing model rather than the ideal analytic path.

In [3]:
shortlist = [
    {"label": "ell0_core3_L150", "ell": 0, "D_um": 3.0, "L_um": 150.0, "Ein_uJ": 10.0},
    {"label": "ell3_core3_L150", "ell": 3, "D_um": 3.0, "L_um": 150.0, "Ein_uJ": 10.0},
    {"label": "ell5_core4_L200", "ell": 5, "D_um": 4.0, "L_um": 200.0, "Ein_uJ": 20.0},
]
bundles = []
for item in shortlist:
    cfg = replace(
        base,
        target=replace(
            base.target,
            ell=item["ell"],
            target_core_diameter_m=item["D_um"] * bt.um,
            target_bessel_length_m=item["L_um"] * bt.um,
        ),
        energy=replace(base.energy, pulse_energy_in_J=item["Ein_uJ"] * bt.uJ),
    )
    bundle = pdiag.build_case_bundle(cfg, preset=PRESET, path=PATH, case_id=item["label"])
    bundle["input_energy_uJ"] = item["Ein_uJ"]
    bundles.append(bundle)
summary = pdiag.shortlist_summary_dataframe(bundles)
summary.to_csv(PUB_CSV_DIR / "shortlist_realistic_summary.csv", index=False)
display(summary)

,case_id,path,ell,target_core_diameter_um,target_bessel_length_um,gamma_slm_deg,magnification_to_sample,ring_radius_um,core_radius_um,feature_diameter_um,...,bessel_region_um,bessel_region_start_um,bessel_region_end_um,line_proxy_threshold_length_um,qa_status,phase_sampling_label,focal_sampling_label,axial_sampling_label,focal_kt_nyquist_over_kr,input_energy_uJ
0,ell0_core3_L150,realistic,0,3.0,150.0,0.592539,0.008071,0.000000,0.674853,1.349706,...,37.631884,-81.929850,-44.297966,0.0,marginal,pass,pass,pass,7.799569,10.0
1,ell3_core3_L150,realistic,3,3.0,150.0,0.592539,0.008071,1.635420,1.157494,3.270841,...,38.015567,-81.334411,-43.318844,0.0,marginal,pass,pass,pass,7.799569,10.0
2,ell5_core4_L200,realistic,5,4.0,200.0,0.444411,0.008071,3.148242,2.283406,6.296483,...,36.330855,-64.316941,-27.986086,0.0,marginal,pass,pass,pass,10.399426,20.0


## Detailed Case Review

Pick one shortlist case for a deeper publication diagnostic panel, energy-budget review, and axial plane montage.

In [4]:
CASE_LABEL = "ell3_core3_L150"
selected = next(bundle for bundle in bundles if bundle["case_id"] == CASE_LABEL)
display(selected["summary"])
display(selected["energy_budget"])

,metric,value
0,case_id,ell3_core3_L150
1,path,realistic
2,ell,3
3,target_core_diameter_um,3.0
4,target_bessel_length_um,150.0
5,gamma_slm_deg,0.592539
6,magnification_to_sample,0.008071
7,ring_radius_um,1.63542
8,core_radius_um,1.157494
9,feature_diameter_um,3.270841


,stage_order,stage,incremental_transmission,cumulative_transmission,pulse_energy_uJ
0,0,Input pulse,1.00,1.000000,10.00000
1,1,Pre-SLM optics,1.00,1.000000,10.00000
2,2,SLM reflectivity,0.75,0.750000,7.50000
3,3,First-order efficiency,0.45,0.337500,3.37500
4,4,Relay transmission,0.90,0.303750,3.03750
5,5,Focusing transmission,0.90,0.273375,2.73375
6,6,Sample surface,0.96,0.262440,2.62440
7,7,User transmission,1.00,0.262440,2.62440


In [5]:
fig = pdiag.plot_case_diagnostics(selected)
plt.show()

C:\Users\sm2006\AppData\Local\Temp\ipykernel_19252\1477822169.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig = pdiag.plot_axial_plane_montage(selected)
plt.show()

C:\Users\sm2006\AppData\Local\Temp\ipykernel_19252\2418045266.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Shortlist Metric Comparison

Quick bar plots for the shortlist so the case selection logic is visible in the same notebook as the figures.

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
region_col = "bessel_region_um" if "bessel_region_um" in summary else "bessel_zone_um"
ax[0].bar(summary["case_id"], summary[region_col], color=vbb_style.CATEGORICAL_PALETTE[2])
ax[0].plot(summary["case_id"], summary["bessel_zone_um"], "o", color="0.2", label="legacy FWHM zone")
ax[0].set_title("Stable Bessel region")
ax[0].set_ylabel("z length [um]")
ax[0].tick_params(axis="x", rotation=25)
ax[0].legend(frameon=False)
ax[1].bar(summary["case_id"], summary["peak_fluence_J_cm2"], color=vbb_style.CATEGORICAL_PALETTE[1])
ax[1].set_title("Peak fluence")
ax[1].set_ylabel("J/cm^2")
ax[1].tick_params(axis="x", rotation=25)
ax[2].bar(summary["case_id"], summary["side_to_core_peak_ratio"], color=vbb_style.CATEGORICAL_PALETTE[0])
ax[2].set_title("Side-to-core peak ratio")
ax[2].set_ylabel("ratio")
ax[2].tick_params(axis="x", rotation=25)
plt.show()

C:\Users\sm2006\AppData\Local\Temp\ipykernel_19252\3451469057.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export Figures

Save the selected-case figures into the publication-study output subtree so they are reproducible and easy to collect for manuscript drafting.

In [8]:
diag_path = PUB_FIG_DIR / f"{CASE_LABEL}_diagnostics.png"
montage_path = PUB_FIG_DIR / f"{CASE_LABEL}_plane_montage.png"
fig1 = pdiag.plot_case_diagnostics(selected)
fig1.savefig(diag_path, dpi=220, bbox_inches="tight")
plt.close(fig1)
fig2 = pdiag.plot_axial_plane_montage(selected)
fig2.savefig(montage_path, dpi=220, bbox_inches="tight")
plt.close(fig2)
print(diag_path)
print(montage_path)

C:\PhD\Code\Publication_Study\Publication_Study\outputs\figures\publication_study\ell3_core3_L150_diagnostics.png
C:\PhD\Code\Publication_Study\Publication_Study\outputs\figures\publication_study\ell3_core3_L150_plane_montage.png
